# Building the agent system

### Everything on this screen is running for real. Nothing here is a slide.

---

In notebook 1 you built the thing that **notices**. It can tell you that
`surge_coverage_pct` is 75.33% when a person decided it must be 100%.

It cannot tell you **why**, and that is where the hours actually go. Somebody has
to open the run log, read the quarantine, read the pipeline source, find the
release, write it up, raise the ticket, and propose the fix.

That work is a **method**. This notebook builds the thing that follows it.

You will build an agent from nothing: a tool, a loop, a typed answer. Then five
of them, then the thing that decides who is asked next, and then the guards that
mean none of it can change a number.

In [ ]:
import sys; sys.path.insert(0, '..')
import inspect, json, psycopg
from pipelines.lib.config import dsn, SCHEMA
from nb import show, sql, fetch, run

print('connected.')

### Start from a clean slate, then break it

We need a real problem for the agents to work on. This resets everything, then
applies the same break as notebook 1: the mobile team moves the surge field.

**Safe to run at any point.** If you get lost later, come back here.

In [ ]:
run('break_it.py', '--fix')
run('cli.py', 'reset')
run('cli.py', 'run', 'all')

In [ ]:
run('break_it.py')                       # move the surge field
run('cli.py', 'run', 'p3_bronze_driver_app')
run('cli.py', 'run', 'p7_silver_rides')
run('cli.py', 'run', 'p8_gold_daily')
run('cli.py', 'signals')

`surge_coverage_pct` is red. Now we build the thing that works out why.

---

# Part 1 · What an agent actually is

Three ideas. That is genuinely all of it, and none of them are hard.

## Idea 1 · A tool is a python function

Not a plugin. Not a connector. A function, with a docstring, and the docstring is
the API: it is the only thing the model reads to decide whether to call it.

In [ ]:
from langchain.tools import tool

@tool
def rows_in_table(table: str) -> str:
    """How many rows are in one table of the warehouse.

    Use this to check whether a table is empty or unexpectedly small.
    """
    with psycopg.connect(dsn()) as c:
        n = c.execute(f'SELECT count(*) FROM {SCHEMA}.{table}').fetchone()[0]
    return f'{table} has {n:,} rows'

print('name       ', rows_in_table.name)
print('description', rows_in_table.description.splitlines()[0])
print()
print(rows_in_table.invoke({'table': 'bronze_driver_app'}))

> **Write the docstring for the model, not for a colleague.** It is the only
> documentation the model gets. A vague one produces a tool that is never called,
> or worse, called for the wrong thing.

## Idea 2 · An agent is a loop

Give a model some tools and a question. It picks a tool, sees the result, decides
what to do next, and stops when it has an answer.

That loop is the entire idea. Everything else is engineering around it.

In [ ]:
from langchain.agents import create_agent

little = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[rows_in_table],
    system_prompt='You answer questions about the warehouse using the tools you '
                  'have. Quote the real numbers you were given.',
)

out = little.invoke({'messages': [{'role': 'user', 'content':
    'How many rows are in bronze_driver_app and in silver_rides?'}]})

print(out['messages'][-1].content)

### Now look at the loop itself

The answer is not the interesting part. **This** is.

In [ ]:
for m in out['messages']:
    kind = type(m).__name__.replace('Message', '')
    if getattr(m, 'tool_calls', None):
        for call in m.tool_calls:
            print(f'  {kind:9}  CALL   {call["name"]}({call["args"]})')
    elif kind == 'Tool':
        print(f'  {kind:9}  RESULT {m.content}')
    elif m.content:
        print(f'  {kind:9}  {str(m.content)[:90]}')

Read that trace top to bottom. Nobody wrote *"call rows_in_table twice"*. The
model was given a question and a toolbox and worked out the order.

**That is the whole of what an agent is.** A loop, a toolbox, and a stopping
condition.

## Idea 3 · Make it answer in fields, not paragraphs

An agent that replies *"it looks like the driver app data might be incomplete"*
is useless to a program. You cannot branch on it, count it, or put it in a
column.

In [ ]:
from pydantic import BaseModel, Field

class TableCheck(BaseModel):
    table:     str  = Field(description='the table you looked at')
    row_count: int  = Field(description='how many rows it has')
    is_empty:  bool = Field(description='true if it has no rows at all')
    comment:   str  = Field(description='one sentence a person can read')

typed = create_agent(model='openai:gpt-5.4-mini', tools=[rows_in_table],
                     system_prompt='Check the table you are asked about.',
                     response_format=TableCheck)

v = typed.invoke({'messages': [{'role': 'user', 'content':
    'Check bronze_driver_app.'}]})['structured_response']

print(type(v).__name__)
for k, val in v.model_dump().items():
    print(f'  {k:10} {val}')

`v.is_empty` is a real boolean. The supervisor you are about to build branches on
exactly this kind of field.

> **A supervisor cannot make a decision from a paragraph.**

### That is the whole of it

```
    a tool          a python function with a docstring
    an agent        a model, some tools, and a loop
    a verdict       a pydantic model, so a program can use the answer
```

Everything from here is about **what each agent is not allowed to do**.

---

# Part 2 · Why five agents, and not one

The obvious design is one agent with every tool. It is also the one that stops
working the first time the problem is unfamiliar.

> **An agent with every tool will use every tool.**

Give one agent SQL *and* file reading and it will read a file, form a theory, and
then go looking for numbers that agree with it. That is not an investigation,
that is confirmation, and it is convincing precisely when it is wrong.

## So each one gets a question and a toolbox, and nothing else

| | agent | the question it answers | what it can touch |
|---|---|---|---|
| 1 | **triage** | is this real, or is it Sunday? | the board only |
| 2 | **data detective** | what is wrong with the rows? | read-only SQL |
| 3 | **lineage detective** | where did it enter? | source and git |
| 4 | **remediation** | what is the smallest fix? | writes artifacts |
| 5 | **verifier** | did it actually work? | runs pipelines and checks |

In [ ]:
from agent_service.tools.warehouse import READ_TOOLS
from agent_service.tools.lineage import LINEAGE_TOOLS
from agent_service.tools.signals import SIGNAL_TOOLS
from agent_service.tools.publish import PUBLISH_TOOLS
from agent_service.tools.verify import VERIFY_TOOLS

for label, tools in [('1 triage', SIGNAL_TOOLS), ('2 data detective', READ_TOOLS),
                     ('3 lineage detective', LINEAGE_TOOLS),
                     ('4 remediation', PUBLISH_TOOLS), ('5 verifier', VERIFY_TOOLS)]:
    print(f'{label}')
    for t in tools:
        print(f'      {t.name}')
    print()

### Read rows two and three again

**The data detective cannot open a file.** So everything it reports came from a
query. It cannot guess from the code and present the guess as evidence.

**The lineage detective cannot query the warehouse.** So it cannot quietly redo
the previous agent's work with worse tools.

> **The method emerges from the constraint, not from a longer prompt that
> everybody hopes the model reads.**

## And triage is cheap, and runs first, on purpose

In [ ]:
from agent_service.agents import TRIAGE_PROMPT
print(TRIAGE_PROMPT)

If triage says the number moved for a boring reason, **nothing else runs**. Four
expensive agents are never woken and nobody is paged for a public holiday.

> **Be willing to say no. A triage agent that says yes to everything has cost you
> the money it was there to save.**

## Watch it say no

Give it a signal that has not actually breached.

In [ ]:
from agent_service.agents import triage_agent
from signal_service import evaluate as ev
from signal_service.kpis import get

kpi = get('events_per_ride')            # healthy
reading, verdict = ev.evaluate(kpi)
print(f'{kpi.name}: {verdict.value} vs {verdict.baseline}, breached={verdict.breached}')
print()

v = triage_agent().invoke({'messages': [{'role': 'user', 'content':
    f'The signal {kpi.name} is at {verdict.value}{kpi.unit}, normally '
    f'{verdict.baseline}{kpi.unit}. It means: {kpi.means}. Is this real?'
}]})['structured_response']

print('is_real ', v.is_real)
print('reason  ', v.reason)

## Every prompt is a method, not a list of answers

This is the difference between a system that works on a problem nobody has seen
and one that recognises the three incidents somebody wrote down.

In [ ]:
from agent_service.agents import DATA_PROMPT
print(DATA_PROMPT)

Read it again. It contains **an order to look in** and **a standard of evidence**.
Nowhere does it say *"if surge is missing, look at app_version"*. There is no
list of incidents anywhere in this project.

Note the paragraph about held records. It is there because the model kept
reporting the wrong app version: the rows that explain this incident **never
landed in any table**, so no query over the warehouse can see them. They are in
quarantine, and the held payload is the only place the truth is written down.

> **The prompt teaches where to look. The tools decide what is reachable. Neither
> one contains the answer.**

---

# Part 3 · The thing that holds the plan

Five specialists is not a system. Something has to decide who is asked next and
carry each answer forward.

The documented LangChain pattern for this is **subagents as tools**: each
specialist is an agent, wrapped with `@tool`, and one main agent holds all five.
The older `create_supervisor` helper is no longer maintained; this replaced it.

In [ ]:
from agent_service import agents

src = inspect.getsource(agents.build_subagent_tools)
print(src[src.index('    @tool("triage"'):src.index('    @tool("investigate_lineage"')])

Each wrapper does three things: run the specialist, take its typed verdict, hand
back JSON the supervisor can read.

The specialists are **stateless**. Each starts in a clean context every time,
which is what stops the fifth agent inheriting four agents' worth of noise.

## And the supervisor holds the plan and nothing else

In [ ]:
from agent_service.supervisor import SUPERVISOR_PROMPT
print(SUPERVISOR_PROMPT)

### What it does not do

It does not investigate. It decides who is asked next, carries each answer
forward, and stops when the evidence is enough.

**It stops early too.** That is the `if triage says the breach is not real, STOP`
rule, and it is the difference between a system that costs a few cents a night
and one that costs a few hundred.

---

# Part 4 · What stops it doing something stupid

This is the part that makes the difference between a demo and something you would
actually run against a warehouse.

## The rule

> **Whether an agent meant well is a judgement.**
> **Whether it CAN write is a fact.**

Every boundary in this system is a fact.

## 1 · The read-only tool cannot write, and not because we asked

In [ ]:
from agent_service.tools.warehouse import run_sql

# @tool wraps the function in a StructuredTool, so the original is .func
print(inspect.getsource(run_sql.func))

There are two guards there and only one of them counts.

The regex is a **courtesy**: it returns a clear message instead of a database
error, and it stops an honest mistake early. If it were the only guard, this
system would be one clever string away from a deleted table.

`conn.read_only = True` is the one that counts. **Postgres** refuses the write.
Not our code, not our regex, not the model's good intentions.

In [ ]:
print(run_sql.invoke({'query': 'DELETE FROM teach.silver_rides'}))
print()
print(run_sql.invoke({'query': 'UPDATE teach.gold_daily SET revenue = 0'}))

## 2 · A code change happens in an isolated checkout, and never on your branch

In [ ]:
from agent_service.tools.repo import WRITABLE, pushing_allowed
print('the agent may only propose changes to:')
for pattern in WRITABLE:
    print(f'    {pattern}')
print()
print('pushing allowed right now:', pushing_allowed())

Two more facts underneath that list:

**It works in a `git worktree`**, a separate checkout of the same repository. Two
investigations running at once cannot see each other's files. Without it, the
second one commits the first one's changes, reports success, and the branch does
not contain the fix.

**It never merges.** It opens a branch and a pull request. A human decides.

## 3 · A change that would not compile never reaches a human

In [ ]:
from agent_service.tools import repo

src = inspect.getsource(repo.propose)
i = src.index('    # Whether a change is right is a judgement')
print(src[i:src.index('    out.diff =')])

## 4 · Anything that would touch data stops and waits for a person

In [ ]:
from agent_service.tools.publish import request_db_change, GATED
print('tools that pause for a human:', GATED)
print()
print(request_db_change.invoke({
    'breach_id': 'BRCDEMO',
    'summary': 'backfill the missing surge values',
    'rationale': 'the pricing report is wrong until these rows have a value',
    'statement': 'UPDATE teach.silver_rides SET surge = 1.0 WHERE surge IS NULL',
    'rows_affected_estimate': 'about 9,900 rows, counted with a SELECT first',
    'reversible': 'yes: rebuild silver from bronze, which is untouched',
}))

**It did not run that statement, and nothing in this system will.** The tool
records what was proposed, what it would touch and how to reverse it, and stops.

That is `HumanInTheLoopMiddleware`, and the investigation genuinely pauses:
`waiting_for_human` comes back true, and a person answers it with
`POST /investigations/{id}/resume`. **Rejecting is a real outcome**, and the note
goes back to the agent. That is the difference between a human gate and a rubber
stamp.

## The four boundaries, and how each is enforced

| boundary | enforced by | not by |
|---|---|---|
| cannot write to the warehouse | Postgres, `read_only = True` | a prompt |
| cannot edit files outside a whitelist | a path check, in an isolated checkout | a prompt |
| cannot merge anything | opening a pull request instead | a prompt |
| cannot touch data | an interrupt that pauses the run | a prompt |

And every one of them has a test that tries to break it.

In [ ]:
run('-m', 'pytest', 'tests/test_guards.py', '-q', '--no-header', '--color=no')

---

# Part 5 · Running it, and watching every step

Now the whole thing, on the break we made at the top. Five agents, one incident,
no human.

**Run this in a terminal, not in this cell.** The notebook captures the output
and prints it when it is finished; the terminal streams it, so the room watches
the method rather than reading a conclusion.

```
python cli.py investigate surge_coverage_pct
```

You will see, live:

```
  1 TRIAGE              is this real?
      get_signal_board()
      get_kpi_definition(name=surge_coverage_pct)
      -> triage answered      is_real True   severity high

  2 DATA DETECTIVE      what is wrong with the rows?
      recent_runs(limit=10)
      quarantine_sample(pipeline=p3_bronze_driver_app, limit=5)
      -> data_detective answered

  3 LINEAGE DETECTIVE   where did it enter?
      read_source(path=pipelines/p3_bronze_driver_app.py)
      -> lineage_detective answered

  4 REMEDIATION         what is the smallest fix?
      propose_code_change(...)     -> a real pull request
      raise_ticket(...)
      publish_incident_page(...)

  5 VERIFIER            did it actually work?
      check_file_on_disk(...)
      -> resolved False, change_applied False
```

If you would rather run it here, this cell does the same thing and prints the
whole trace at the end.

In [ ]:
run('cli.py', 'investigate', 'surge_coverage_pct')

## What just happened, in order

1. **Triage** read the board and said it was real, and said why
2. **The data detective** queried, found the held records, and quoted the numbers
3. **The lineage detective** read `p3_bronze_driver_app.py` and found the contract
4. **Remediation** opened a branch and a pull request, raised a ticket, and wrote
   the Confluence page
5. **The verifier** checked the file on disk and said plainly: **not applied, still
   on a branch, nothing is fixed**

**Nobody typed anything after the break.** And there is no list of incidents
anywhere in this project: the agents were given a method and a toolbox, which is
why a problem nobody has seen before is worked exactly like this one.

## What it produced, and what it did not

Everything the investigation leaves behind is **for a person to decide on**.

In [ ]:
sql("""
    SELECT ticket_id, kind, severity, left(title, 60) AS title
    FROM oncall.tickets ORDER BY raised_at DESC LIMIT 5
""", 'tickets raised')

sql("""
    SELECT request_id, kind, status, coalesce(pr_url, '(no remote)') AS pull_request
    FROM oncall.change_requests ORDER BY created_at DESC LIMIT 5
""", 'code changes proposed · none of them merged')

The verifier said `resolved: False`, and it is right. **A proposal waiting for a
human has fixed nothing**, and an agent that reports otherwise is worse than no
agent, because the incident looks handled and is not.

---

# What to take away

> ### An agent is a loop, a toolbox, and a stopping condition. Nothing more.
> ### Five narrow agents beat one wide one, because the constraint enforces the method.
> ### Make it answer in fields, because a supervisor cannot branch on a paragraph.
> ### If a boundary matters, make it something the system CANNOT do, not something you asked it not to.

## Put it back

Clears the break, the incidents, the tickets and the agent's branches, so both
notebooks are ready to run again from the top.

In [ ]:
run('break_it.py', '--fix')     # undo the break, if it is still in place
run('cli.py', 'reset')          # empty warehouse, no incidents, no artifacts
run('cli.py', 'run', 'all')     # rebuild all eight pipelines, about 40 seconds

---

### Where to go next

| | |
|---|---|
| notebooks **1 to 12** | build the warehouse yourself, one source at a time |
| notebooks **13 to 18** | the same two services, in more depth |
| notebook **19** | a guided tour of the source, in dependency order |
| `docs/` | twelve documents, one per idea |

Every file in the project opens with a docstring explaining the **decision**
rather than the syntax. Those docstrings are the real material.